In [ ]:
import os
import json
import pickle
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELSDIR  = CONFIGS['filepaths']['models']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SUBSETFRAC = CONFIGS['experiments']['sr']['subsetfrac']
SPLIT      = 'train'

with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
TPMEAN = STATS['tp_mean']
TPSTD  = STATS['tp_std']
ZMIN   = (0.0-TPMEAN)/TPSTD

REGISTRY = os.path.join(MODELSDIR,'sr','optimized_equations.pkl')
with open(REGISTRY,'rb') as f:
    OPTIMEQS = pickle.load(f)
SRATMCONSTS = OPTIMEQS['sr_atm_eq']['constants']
SRATMFORM   = OPTIMEQS['sr_atm_eq']['form']
print(f'SR-ATM form: {SRATMFORM}')
print(f'SR-ATM constants: {SRATMCONSTS}')

In [ ]:
SRFN = dict(cube=lambda x:x**3,square=lambda x:x**2,neg=lambda x:-x,
            exp=np.exp,log=np.log,abs=np.abs,sqrt=np.sqrt,
            max=np.maximum,min=np.minimum)

def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None: w *= mask[:,None,:]
    return w.sum(axis=2)

def eval_form(form,feats,constants=None):
    ns = dict(SRFN,__builtins__={},**feats)
    if constants is not None: ns.update(constants)
    return np.asarray(eval(form,ns),dtype=float)

def to_mm_from_z(z):
    return np.maximum(np.expm1((ZMIN+np.maximum(z,0.0))*TPSTD+TPMEAN),0.0)

def to_z_from_mm(mm):
    return (np.log1p(np.maximum(mm,0.0))-TPMEAN)/TPSTD

def to_da(arr,ref):
    return xr.DataArray(arr.reshape(ref.shape),dims=ref.dims,coords=ref.coords)

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = ds.time.size,ds.lat.size,ds.lon.size
    nsig  = ds.sizes.get('sig',1)
    dsig  = ds.dsig.values
    refda = ds.tp.transpose('time','lat','lon').load()
    target = refda.values.ravel()
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds.surfmask.transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    lf = flat('lf')

kernels = [xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{s}_weights.nc'),engine='h5netcdf')['k'].values for s in SEEDS]
integ   = kernel_integrate(fields,np.mean(kernels,axis=0),dsig,surfmask)
FEATS   = {v:integ[:,i] for i,v in enumerate(FIELDVARS)}

sratmpred = eval_form(SRATMFORM,FEATS,SRATMCONSTS)
sratmmm   = to_mm_from_z(sratmpred)

nnpath = os.path.join(PREDSDIR,f'nn_gauss_{SPLIT}_predictions.nc')
with xr.open_dataset(nnpath) as ds:
    nnpred = ds.tp.load()
if 'seed' in nnpred.dims: nnpred = nnpred.mean('seed')
nnpred = nnpred.transpose('time','lat','lon')
nnmm   = nnpred.values.ravel()
nnz    = to_z_from_mm(nnmm)

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsmm = ds.tp.transpose('time','lat','lon').load().values.ravel()

print(f'Loaded {SPLIT} split: {ntime} times, {nlat} lats, {nlon} lons = {len(target):,} points')
print(f'SR-ATM z-scored pred range: [{sratmpred.min():.3f}, {sratmpred.max():.3f}]')
print(f'NN-GAUSS z-scored pred range: [{nnz.min():.3f}, {nnz.max():.3f}]')

In [ ]:
disagreement = np.abs(nnmm-sratmmm)
sratmerr     = (obsmm-sratmmm)**2
nnerr        = (obsmm-nnmm)**2
skillgap     = sratmerr-nnerr

residual     = target-sratmpred
absresidual  = np.abs(residual)

print('=== Error metrics (training split) ===')
for name,vals in [('|NN-SR_ATM| (mm)',disagreement),
                  ('SR-ATM MSE (mm²)',sratmerr),
                  ('NN-GAUSS MSE (mm²)',nnerr),
                  ('Skill gap (mm²)',skillgap),
                  ('|residual| (z-scored)',absresidual)]:
    finite = vals[np.isfinite(vals)]
    print(f'{name:30s}  mean={finite.mean():.4f}  p50={np.median(finite):.4f}  '
          f'p90={np.percentile(finite,90):.4f}  p99={np.percentile(finite,99):.4f}')

In [ ]:
disag_da   = to_da(disagreement,refda)
skillgapda = to_da(skillgap,refda)
absresda   = to_da(absresidual,refda)

fig,axs = pplt.subplots(ncols=3,refwidth=2.5,proj='cyl',proj_kw={'lon_0':75})
axs.format(coast=True,lonlim=(60,90),latlim=(5,25),
           lonlocator=10,latlocator=5,grid=False,
           suptitle='Time-Mean Error Metrics (Training Split)')

m0 = axs[0].pcolormesh(refda.lon,refda.lat,disag_da.mean('time'),
                       cmap='Reds',vmin=0)
axs[0].colorbar(m0,loc='b',label='mm')
axs[0].set_title('|NN − SR-ATM|')

vmax = float(np.abs(skillgapda.mean('time')).quantile(0.98))
m1 = axs[1].pcolormesh(refda.lon,refda.lat,skillgapda.mean('time'),
                       cmap='RdBu_r',vmin=-vmax,vmax=vmax)
axs[1].colorbar(m1,loc='b',label='mm²')
axs[1].set_title('Skill Gap (SR-ATM err² − NN err²)')

m2 = axs[2].pcolormesh(refda.lon,refda.lat,absresda.mean('time'),
                       cmap='Reds',vmin=0)
axs[2].colorbar(m2,loc='b',label='z-scored')
axs[2].set_title('|SR-ATM Residual|')

pplt.show()

In [ ]:
corrs = np.corrcoef(np.stack([disagreement,absresidual,np.maximum(skillgap,0)]))[np.triu_indices(3,k=1)]
print('Pairwise correlations between error metrics:')
print(f'  |NN-SR_ATM| vs |residual|:   {corrs[0]:.3f}')
print(f'  |NN-SR_ATM| vs skill_gap+:   {corrs[1]:.3f}')
print(f'  |residual|  vs skill_gap+:   {corrs[2]:.3f}')
print()
print('If these are highly correlated, |residual| is a good proxy for the NN–SR-ATM gap.')
print('This matters because |residual| is already available during training')
print('(no need to load NN predictions), simplifying the pipeline.')

In [ ]:
landmask  = lf>0.5
oceanmask = lf<0.5

fig,axs = pplt.subplots(ncols=2,refwidth=2.8,refheight=2.5)
axs.format(grid=False,xlabel='|NN − SR-ATM| (mm)',ylabel='Density')

bins = np.linspace(0,float(np.percentile(disagreement,99.5)),80)
axs[0].hist(disagreement[landmask],bins=bins,density=True,alpha=0.7,color='#a05a2c',label='Land')
axs[0].hist(disagreement[oceanmask],bins=bins,density=True,alpha=0.7,color='#2a7fbf',label='Ocean')
axs[0].legend(loc='ur')
axs[0].set_title('Model Disagreement by Surface')

precipbins = [0,0.1,1,5,20,999]
binlabels  = ['Dry (<0.1)','Light (0.1–1)','Moderate (1–5)','Heavy (5–20)','Extreme (>20)']
colors     = pplt.Colormap('viridis')(np.linspace(0.15,0.85,len(binlabels)))
for i in range(len(precipbins)-1):
    mask = (obsmm>=precipbins[i])&(obsmm<precipbins[i+1])
    if mask.sum()==0: continue
    axs[1].hist(disagreement[mask],bins=bins,density=True,alpha=0.6,
                color=colors[i],label=binlabels[i])
axs[1].legend(loc='ur',fontsize=7)
axs[1].set_title('Model Disagreement by Precip Regime')
pplt.show()

In [ ]:
rng = np.random.default_rng(42)
n   = len(target)
nsamp = max(1,int(round(SUBSETFRAC*n)))

def weighted_sample(weights,nsamp,rng):
    prob = weights/weights.sum()
    return rng.choice(len(weights),nsamp,replace=False,p=prob)

uniform_idx = rng.choice(n,nsamp,replace=False)

ALPHAS = [0,2,5,10,20]
samples = {}
for alpha in ALPHAS:
    w = 1.0+alpha*(disagreement/(np.percentile(disagreement,99)+1e-12)).clip(max=1.0)
    idx = weighted_sample(w,nsamp,np.random.default_rng(42))
    samples[alpha] = idx

print(f'Total points: {n:,}')
print(f'Subsample size ({SUBSETFRAC:.1%}): {nsamp:,}')
print()
print(f'{"alpha":>6s}  {"mean |disag|":>12s}  {"p90 |disag|":>12s}  '
      f'{"frac high-err":>14s}  {"frac land":>10s}')
p90thresh = np.percentile(disagreement,90)
for alpha in ALPHAS:
    idx = samples[alpha]
    d   = disagreement[idx]
    print(f'{alpha:6d}  {d.mean():12.4f}  {np.percentile(d,90):12.4f}  '
          f'{(d>p90thresh).mean():14.3f}  {landmask[idx].mean():10.3f}')

In [ ]:
fig,axs = pplt.subplots(ncols=3,nrows=2,refwidth=2.2,refheight=1.8,share=False)
bins = np.linspace(0,float(np.percentile(disagreement,99.5)),60)

for i,alpha in enumerate([0,2,5,10,20]):
    row,col = divmod(i,3)
    ax = axs[row,col]
    idx = samples[alpha]
    ax.hist(disagreement,bins=bins,density=True,alpha=0.4,color='gray',label='Full')
    ax.hist(disagreement[idx],bins=bins,density=True,alpha=0.7,color='#4C72B0',label='Sampled')
    ax.set_title(f'α = {alpha}')
    ax.format(grid=False)
    if i==0: ax.legend(loc='ur',fontsize=7)

axs[-1,-1].axis('off')
axs.format(xlabel='|NN − SR-ATM| (mm)',ylabel='Density',
           suptitle='Sampling Distribution vs Full Distribution')
pplt.show()

In [ ]:
resid_weights = absresidual/(np.percentile(absresidual,99)+1e-12)
resid_weights = resid_weights.clip(max=1.0)
disag_weights = disagreement/(np.percentile(disagreement,99)+1e-12)
disag_weights = disag_weights.clip(max=1.0)

ALPHA = 5
resid_idx = weighted_sample(1.0+ALPHA*resid_weights,nsamp,np.random.default_rng(42))
disag_idx = weighted_sample(1.0+ALPHA*disag_weights,nsamp,np.random.default_rng(42))

print(f'Comparing weight sources at alpha={ALPHA}:')
print(f'{"":20s}  {"mean disag":>11s}  {"p90 disag":>10s}  '
      f'{"mean |resid|":>13s}  {"p90 |resid|":>12s}  {"frac land":>10s}')
for name,idx in [('Uniform',uniform_idx),
                 ('|residual| wt',resid_idx),
                 ('|NN-SR_ATM| wt',disag_idx)]:
    d = disagreement[idx]
    r = absresidual[idx]
    print(f'{name:20s}  {d.mean():11.4f}  {np.percentile(d,90):10.4f}  '
          f'{r.mean():13.4f}  {np.percentile(r,90):12.4f}  {landmask[idx].mean():10.3f}')
print()
print('If |residual|-weighted sampling achieves similar bias as |NN-SR_ATM|-weighted,')
print('it is the simpler choice: no NN prediction files needed at training time.')

In [ ]:
fig,axs = pplt.subplots(ncols=3,refwidth=2.5,proj='cyl',proj_kw={'lon_0':75})
axs.format(coast=True,lonlim=(60,90),latlim=(5,25),
           lonlocator=10,latlocator=5,grid=False,
           suptitle=f'Sampling Density Maps (α={ALPHA})')

def sample_density(idx,ref):
    counts = np.zeros(len(target))
    np.add.at(counts,idx,1)
    return to_da(counts,ref).mean('time')

for i,(name,idx) in enumerate([('Uniform',uniform_idx),
                               ('|Residual| Weighted',resid_idx),
                               ('|NN−SR-ATM| Weighted',disag_idx)]):
    density = sample_density(idx,refda)
    m = axs[i].pcolormesh(refda.lon,refda.lat,density,cmap='Reds',vmin=0)
    axs[i].colorbar(m,loc='b',label='Samples per grid cell')
    axs[i].set_title(name)

pplt.show()